# Term simplification via rewriting

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "RegExpParser.fsx"

In [ ]:
open AutomataTypes
open RegExpParser

The original notebook represents regular expressions as nested tuples, using uppercase letters as pattern variables. Here the expression type is the same `RegExp` union as in the parser, while **patterns have a separate type**. Thus a real expression containing the literal uppercase character `R` is never mistaken for a variable. `patternOf` interprets uppercase letters as variables only when constructing a rewrite rule.

In [ ]:
type Pattern =
    | PatternVariable of string
    | PatternEmpty
    | PatternEpsilon
    | PatternLiteral of char
    | PatternConcat of Pattern * Pattern
    | PatternAlt of Pattern * Pattern
    | PatternStar of Pattern

type Substitution = Map<string, RegExp>
type Rule = Pattern * Pattern

let rec patternOf expression =
    match expression with
    | Literal c when 'A' <= c && c <= 'Z' -> PatternVariable (string c)
    | Empty -> PatternEmpty
    | Epsilon -> PatternEpsilon
    | Literal c -> PatternLiteral c
    | Concat (left, right) -> PatternConcat (patternOf left, patternOf right)
    | Alt (left, right) -> PatternAlt (patternOf left, patternOf right)
    | Star r -> PatternStar (patternOf r)

A substitution is an immutable map from pattern-variable names to regular expressions. Matching returns `Some substitution` when it succeeds and `None` otherwise. Repeated occurrences of a pattern variable must match exactly the same expression. Each match starts with a fresh map, avoiding the original mutable default argument.

In [ ]:
let rec matchPattern pattern term (substitution: Substitution) : Substitution option =
    match pattern, term with
    | PatternVariable name, _ ->
        match Map.tryFind name substitution with
        | Some previous when previous = term -> Some substitution
        | Some _ -> None
        | None -> Some (Map.add name term substitution)
    | PatternEmpty, Empty | PatternEpsilon, Epsilon -> Some substitution
    | PatternLiteral c, Literal d when c = d -> Some substitution
    | PatternConcat (p1, p2), Concat (t1, t2)
    | PatternAlt (p1, p2), Alt (t1, t2) ->
        matchPattern p1 t1 substitution |> Option.bind (fun next -> matchPattern p2 t2 next)
    | PatternStar p, Star t -> matchPattern p t substitution
    | _ -> None

`apply` instantiates a rule's right-hand side with the bindings from its left-hand side. `rewrite` tries one rule at the root of a term.

In [ ]:
let rec apply pattern (substitution: Substitution) : RegExp =
    match pattern with
    | PatternVariable name ->
        Map.tryFind name substitution
        |> Option.defaultWith (fun () -> invalidArg "substitution" ("Unbound pattern variable: " + name))
    | PatternEmpty -> Empty
    | PatternEpsilon -> Epsilon
    | PatternLiteral c -> Literal c
    | PatternConcat (left, right) -> Concat (apply left substitution, apply right substitution)
    | PatternAlt (left, right) -> Alt (apply left substitution, apply right substitution)
    | PatternStar r -> Star (apply r substitution)

let rewrite term ((left, right): Rule) : RegExp option =
    matchPattern left term Map.empty |> Option.map (apply right)

These are the original 26 identities for alternatives, concatenation and Kleene star. Their order is explicit and deterministic. `R`, `S` and `T` are pattern variables; juxtaposition denotes concatenation.

In [ ]:
let getRules () : Rule list =
    [ "R+∅", "R"
      "∅+R", "R"
      "R+R", "R"
      "𝜀+R*", "R*"
      "R*+𝜀", "R*"
      "𝜀+RR*", "R*"
      "𝜀+R*R", "R*"
      "RR*+𝜀", "R*"
      "R*R+𝜀", "R*"
      "S+ST", "S(𝜀+T)"
      "S+TS", "(𝜀+T)S"
      "∅R", "∅"
      "R∅", "∅"
      "𝜀R", "R"
      "R𝜀", "R"
      "(𝜀+R)R*", "R*"
      "(R+𝜀)R*", "R*"
      "R*(R+𝜀)", "R*"
      "R*(𝜀+R)", "R*"
      "∅*", "𝜀"
      "𝜀*", "𝜀"
      "(𝜀+R)*", "R*"
      "(R+𝜀)*", "R*"
      "R+(S+T)", "(R+S)+T"
      "R(ST)", "(RS)T"
      "(RS*)(𝜀+S)", "RS*" ]
    |> List.map (fun (left, right) -> patternOf (parse left), patternOf (parse right))

`simplifyOnce` first tries a rule at the root, then recursively simplifies its children when no rule applies. `simplify` repeats the operation until the term no longer changes. Rules are parsed once per normalization. This is a simplifier, not a complete canonical form for regular languages.

In [ ]:
let rec simplifyOnce rules term =
    let rewritten = rules |> List.tryPick (fun rule -> rewrite term rule |> Option.filter ((<>) term))
    match rewritten with
    | Some result -> result
    | None ->
        match term with
        | Concat (left, right) -> Concat (simplifyOnce rules left, simplifyOnce rules right)
        | Alt (left, right) -> Alt (simplifyOnce rules left, simplifyOnce rules right)
        | Star r -> Star (simplifyOnce rules r)
        | _ -> term

let simplify term =
    let rules = getRules ()
    let rec normalize expression =
        let next = simplifyOnce rules expression
        if next = expression then expression else normalize next
    normalize term

In [ ]:
let term = parse "(𝜀+a)a*"
checkNotebook (simplify term = parse "a*")
checkNotebook (simplify (parse "R+R") = Literal 'R')
let repeated = patternOf (parse "R+R")
checkNotebook (matchPattern repeated (parse "a+b") Map.empty = None)
simplify (parse "∅*+a𝜀")